In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", None)

column_types = {
    "id": "Int64",
    "osm_id": "str",
    "addr_street": "str",
    "addr_number": "str",
    "ground_floor_group": "str",
    "ground_floor_function": "str",
    "upper_floor_group": "str",
    "upper_floor_function": "str",
    "number_of_storeys": "Int64",
    "is_mixed_use": "boolean",
    "build_year_est": "Int64",
    "arch_style": "str",
    "condition": "Int64",
    "max_height": "float64",
    "is_heritage": "str",
    "source": "str",
    "confidence": "Int64",
    "date_of_checking": "str",
}

In [2]:
data_gdynia_long_cat = pd.read_csv(
    "../../data/processed/fixed/srodmiescie_long_categories.csv",
    sep=";",
    index_col=0,
    dtype=column_types,
)
data_gdynia_long_cat.head()

,addr_street,function_clean,category
id,,,
219,Stefana Żeromskiego,administracja,organizacje publiczne
258,3 Maja,administracja,organizacje publiczne
374,Starowiejska,administracja,organizacje publiczne
495,10 Lutego,administracja,organizacje publiczne
86,Jana z Kolna,administracja,organizacje publiczne


In [ ]:
data_gdynia_rank = data_gdynia_long_cat.groupby("addr_street", as_index=False)[
    "function_clean"
].agg("size")
data_gdynia_rank["rank"] = data_gdynia_rank["size"].rank(
    ascending=False, method="dense"
)
data_gdynia_rank.head(20)

,addr_street,size,rank
0,10 Lutego,64,5.0
1,3 Maja,32,10.0
2,Aleja Jana Pawła II,20,13.0
3,Aleja Marszałka Józefa Piłsudskiego,3,26.0
4,Antoniego Abrahama,124,2.0
5,Antoniego Hryniewickiego,27,12.0
6,Arkadiusza Rybickiego,12,20.0
7,Armii Krajowej,40,8.0
8,Dworcowa,7,23.0
9,Franciszka Żwirki i Stanisława Wigury,19,14.0


In [ ]:
data_gdynia_shares_per_func = (
    data_gdynia_long_cat.groupby(["addr_street", "function_clean"])
    .agg(num_premises=("function_clean", "size"))
    .assign(
        num_total_premises=lambda d: d.groupby("addr_street")["num_premises"].transform(
            "sum"
        )
    )
    .assign(shares_per_premise=lambda d: d["num_premises"] / d["num_total_premises"])
)


In [5]:
data_gdynia_shares_per_func.head()

num_premises  num_total_premises  \
addr_street function_clean                                     
10 Lutego   administracja              1                  64   
            apteka                     1                  64   
            bank                       2                  64   
            bar                        1                  64   
            brak parteru               1                  64   

                            shares_per_premise  
addr_street function_clean                      
10 Lutego   administracja             0.015625  
            apteka                    0.015625  
            bank                      0.031250  
            bar                       0.015625  
            brak parteru              0.015625

In [ ]:
assert np.isclose(
    (data_gdynia_shares_per_func.groupby("addr_street").sum())["shares_per_premise"], 1
).all()

In [ ]:
data_gdynia_shares_per_func["rank"] = data_gdynia_shares_per_func.groupby(
    "addr_street"
)["num_premises"].rank(ascending=False, method="dense")
data_gdynia_shares_per_func = data_gdynia_shares_per_func.loc[
    data_gdynia_shares_per_func["rank"] <= 3, ["num_premises", "rank"]
].sort_values(["addr_street", "rank"], ascending=True)
data_gdynia_shares_per_func.head(20)

num_premises  \
addr_street                         function_clean                             
10 Lutego                           centrum handlowe                       7   
                                    cukiernia                              4   
                                    pustostan                              4   
                                    kebab                                  3   
                                    piekarnia                              3   
                                    sklep spożywczy                        3   
                                    usługi zdrowie                         3   
3 Maja                              mieszkaniowa wielorodzinna             4   
                                    centrum handlowe                       3   
                                    klub muzyczny                          3   
                                    restauracja                            3   
                                    centrum medyczne                       2   
                                    garaż                                  2   
Aleja Jana Pawła II                 restauracja                            8   
                                    handlowa pamiątki                      3   
                                    szkoła żeglarska                       3   
                                    muzeum                                 2   
Aleja Marszałka Józefa Piłsudskiego mieszkaniowa wielorodzinna             1   
                                    restauracja                            1   
                                    urząd miasta                           1   

                                                                rank  
addr_street                         function_clean                    
10 Lutego                           centrum handlowe             1.0  
                                    cukiernia                    2.0  
                                    pustostan                    2.0  
                                    kebab                        3.0  
                                    piekarnia                    3.0  
                                    sklep spożywczy              3.0  
                                    usługi zdrowie               3.0  
3 Maja                              mieszkaniowa wielorodzinna   1.0  
                                    centrum handlowe             2.0  
                                    klub muzyczny                2.0  
                                    restauracja                  2.0  
                                    centrum medyczne             3.0  
                                    garaż                        3.0  
Aleja Jana Pawła II                 restauracja                  1.0  
                                    handlowa pamiątki            2.0  
                                    szkoła żeglarska             2.0  
                                    muzeum                       3.0  
Aleja Marszałka Józefa Piłsudskiego mieszkaniowa wielorodzinna   1.0  
                                    restauracja                  1.0  
                                    urząd miasta                 1.0

In [ ]:
assert (data_gdynia_shares_per_func["rank"] <= 3).all()